In [1]:
import sys
sys.path.append("..")

from source.system import SystemParams

from source.utils import read_csv
from source.policy import PolicyNet

from source.training import train_net

In [ ]:
import torch
import os
import pickle

EPOCHS       = 1000
LR           = 5e-4
WEIGHT_DECAY = 1e-5
BATCH_SIZE   = 500
T_HOR        = 24.0
SEED_BASE    = 123
LAMBDA_COST  = [0.0, 0.1, 0.25, 0.5]
ALPHA_GRID   = 1e-4
DEVICE       = torch.device("cpu")

params = SystemParams()
S0_single = torch.tensor([[0., 0., 100., 0.01]], device=DEVICE)

energy_prices = read_csv("../data/eprice_test.csv")
price_history = energy_prices[:params.T]
future_price = energy_prices[params.T:]

CHECKPOINT_DIR = "../checkpoints/lambda_experiments"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

all_results = {}

for lambda_cost in LAMBDA_COST:

    policy = PolicyNet().to(DEVICE)
    opt = torch.optim.AdamW(policy.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    res = train_net(
        policy=policy,
        optimizer=opt,
        device=DEVICE,
        params=params,
        S0_single=S0_single,
        endog_history=price_history,
        future_endog=future_price,
        epochs=EPOCHS,
        T=T_HOR,
        lambda_cost=lambda_cost,
        alpha_grid=ALPHA_GRID,
        batch_size=BATCH_SIZE,
        seed_base=SEED_BASE,
        verbose=True,
        checkpoint_dir=CHECKPOINT_DIR
    )

    print(f"lambda: {lambda_cost}, best J: {res['best_J']}")

    all_results[lambda_cost] = res

    filename = f"checkpoint_lambda_{str(lambda_cost).replace('.', '_')}.pt"
    path = os.path.join(CHECKPOINT_DIR, filename)

    torch.save({
        "lambda_cost": lambda_cost,
        "policy_state_dict": policy.state_dict(),
        "optimizer_state_dict": opt.state_dict(),
        "result": res,
    }, path)

    torch.save(all_results, os.path.join(CHECKPOINT_DIR, "all_results.pt"))

    with open(os.path.join(CHECKPOINT_DIR, "all_results.pkl"), "wb") as f:
        pickle.dump(all_results, f)

    print(f"Saved: {path}")

[050] J_train=-9.0383 | J_eval(det)=-9.8649
[100] J_train=-10.8069 | J_eval(det)=-11.4050
[150] J_train=-12.2673 | J_eval(det)=-12.5676
[200] J_train=-13.5906 | J_eval(det)=-13.7189
[250] J_train=-14.2244 | J_eval(det)=-14.2465
[300] J_train=-14.6658 | J_eval(det)=-14.5315
[350] J_train=-14.9499 | J_eval(det)=-14.7011
[400] J_train=-15.2041 | J_eval(det)=-14.9116
[450] J_train=-15.4630 | J_eval(det)=-15.1810
[500] J_train=-15.8349 | J_eval(det)=-15.4577
[550] J_train=-16.3606 | J_eval(det)=-15.4986
[600] J_train=-16.3171 | J_eval(det)=-15.6751
[650] J_train=-16.5079 | J_eval(det)=-15.8078
[700] J_train=-16.5550 | J_eval(det)=-15.9244
[750] J_train=-16.7583 | J_eval(det)=-16.0003
[800] J_train=-16.6856 | J_eval(det)=-16.0760
[850] J_train=-16.7014 | J_eval(det)=-16.1131
[900] J_train=-16.6869 | J_eval(det)=-16.1519
[950] J_train=-16.6802 | J_eval(det)=-16.1264
[1000] J_train=-17.0063 | J_eval(det)=-16.2286

Training ready in 434.6s. Epoch: 948. Best J=-17.3436

Saved policies:
 - Best p